# Module 1: Execution Architecture & Microstructure

**Replacing Linear TCA with Dynamic Programming for Non-Linear Square Root Impact**
*Demonstrating the transition from continuous-time academic models to discrete C++ production environments.*

### 1. The Theoretical Baseline (The Math)

The classical Almgren-Chriss framework assumes market impact is a linear function of the trading rate, yielding a closed-form algebraic solution. However, empirical market microstructure reveals that temporary impact scales non-linearly, typically proportional to the square root of the trading rate. 

To model this, we must abandon the closed-form Hamiltonian and define a stochastic control problem that minimizes the expected cost of non-linear impact alongside the variance penalty of holding inventory over time $T$. The continuous-time objective function is:

$$\min_{\{v_t\}} \mathbb{E} \left[ \int_0^T \left( \eta v_t^{3/2} + \lambda \sigma^2 x_t^2 \right) dt \right]$$

subject to the inventory dynamics $\dot{x}_t = -v_t$, where $x_0 = X$ and $x_T = 0$. Here, $v_t$ is the trading rate, $\eta$ is the square-root impact coefficient, $\lambda$ is risk aversion, and $\sigma^2$ is asset volatility.

### 2. The Discretization Reality (The Engineering Flex)

In our C++ execution engine, the continuous integral $\int dt$ does not exist. The trading session is fractured into discrete matching-engine ticks, and order flow is subject to strictly enforced lot sizes. 

Because the $v_t^{3/2}$ term renders the system non-linear, we solve the Hamilton-Jacobi-Bellman (HJB) equation using Dynamic Programming (DP). I discretized the execution window into $N$ intervals of $\Delta t = T / N$ and mapped the continuous inventory $x$ onto a rigid state-space grid of valid exchange lot multiples. The optimal trajectory is calculated via Backward Induction, computing the value function $V(t, x)$ from $t=T$ back to $t=0$.

In [ ]:
// alpha_research_lab/cpp_engine/quant_core.cpp (Abstraction)
// V represents the flattened 2D Value Function table [time_step][inventory_state]
std::vector<double> V(num_steps * num_states, std::numeric_limits<double>::infinity());
std::vector<int> optimal_trajectory(num_steps * num_states, 0);

// Terminal boundary condition: Remaining inventory at T incurs massive penalty
V[(num_steps - 1) * num_states + 0] = 0.0; 

// Backward Induction (Bellman Equation)
for (int t = num_steps - 2; t >= 0; --t) {
    for (int x = 0; x < num_states; ++x) {
        double min_cost = std::numeric_limits<double>::infinity();
        int best_execution_lots = 0;

        for (int n = 0; n <= x; ++n) { 
            double execution_rate = (n * LOT_SIZE) / dt;
            double impact_cost = eta * std::pow(execution_rate, 1.5) * dt;
            double risk_penalty = lambda * variance * std::pow(x * LOT_SIZE, 2) * dt;
            
            double total_cost = (impact_cost + risk_penalty) + V[(t + 1) * num_states + (x - n)];
            
            if (total_cost < min_cost) {
                min_cost = total_cost;
                best_execution_lots = n;
            }
        }
        V[t * num_states + x] = min_cost;
        optimal_trajectory[t * num_states + x] = best_execution_lots;
    }
}

### 3. The Microstructure Friction (The Institutional Edge)

A mathematically sound DP matrix is useless if it violates the physical mechanics of the exchange. In Asian commodity markets, the textbook assumptions of infinite liquidity and continuous trading disintegrate. The math assumes we *can* execute; the engineering assumes we *might not be allowed to*.

**Theory vs. C++ Reality**

| Friction Point | Academic Assumption (Theory) | C++ Engineering Reality |
| :--- | :--- | :--- |
| **Market Impact** | $\eta$ is a static scalar parameter. | $\eta$ is dynamically recalculated tick-by-tick based on L2 resting liquidity. |
| **Execution Volume** | Executes fractional shares. | Constrained strictly by `LOT_SIZE`. The control space loop iterates in exchange-approved integers. |
| **Liquidity State** | Continuous, uninterrupted liquidity. | **Chinese Limit-Board Locks:** If the asset hits a 9.9% circuit breaker, the C++ engine overrides the DP trajectory, halting execution entirely to prevent passive queuing. |
| **State-Space Bounds**| Infinite depth. | Top-of-book volume constraints prevent blowing through the book and triggering adverse selection. |